# 05 - Fine-tuning the Embedding Model

Teaching `bge-small-en-v1.5` the language of annual reports, using question -> passage pairs
generated from the reports themselves. No manual labels needed.

**Fair test:** the model trains only on the FY2023 and FY2025 reports, and never sees any page that
holds an answer to a golden-set question. The FY2024 report is completely unseen.

**Runtime: T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

## Setup

In [ ]:
# Installing libraries
!pip install -q -U sentence-transformers datasets accelerate google-genai chromadb rank_bm25

In [ ]:
# Mounting Drive, checking the GPU and loading saved files
from google.colab import drive
drive.mount('/content/drive')

import json, re, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import torch, chromadb
from tqdm.auto import tqdm
from IPython.display import display

assert torch.cuda.is_available(), "No GPU: switch to Runtime → Change runtime type → T4 GPU"
device = "cuda"

PROJECT_DIR = Path("/content/drive/MyDrive/annual-report-rag")
EVAL_DIR = PROJECT_DIR / "eval"
MODEL_DIR = PROJECT_DIR / "models" / "bge-small-annual-reports"

docstore = json.loads((PROJECT_DIR / "docstore.json").read_text())

# loading the exact text that was embedded for every item
CHROMA_LOCAL = f"/content/chroma_{int(time.time())}"
shutil.copytree(PROJECT_DIR / "chroma", CHROMA_LOCAL)
col = chromadb.PersistentClient(path=CHROMA_LOCAL).get_collection("annual_reports")
_all = col.get(include=["documents"])
doc_text = dict(zip(_all["ids"], _all["documents"]))
ids = _all["ids"]

# loading the golden set and the relevance judgments from Notebook 3
golden = pd.read_csv(EVAL_DIR / "golden.csv")
golden["gold_pages"] = golden.gold_pages.map(
    lambda s: [int(x) for x in str(s).split(";") if x and x != "nan"])
golden["gold_doc_id"] = golden.gold_doc_id.fillna("")
answerable = golden[golden.qtype != "unanswerable"].reset_index(drop=True)
rel = json.loads((EVAL_DIR / "relevance_cache.json").read_text())
print(f"{len(ids)} items | {len(answerable)} answerable questions | {len(rel)} relevance judgments")

In [ ]:
# Rebuilding query parsing, BM25 and fusion (same as Notebooks 3 and 4)
from rank_bm25 import BM25Okapi

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
COMPANIES = sorted({d["company"] for d in docstore.values()})
YEARS = sorted({d["year"] for d in docstore.values()})
ALIASES = {c: [c.replace("_", " ")] for c in COMPANIES}
ALIASES.setdefault("au_small_finance_bank", []).extend(["au bank", "au sfb"])

def find_years(q):
    ys, q = set(), q.lower()
    for m in re.finditer(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", q):
        end = m.group(2); ys.add(end if len(end) == 4 else "20" + end)
    q = re.sub(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", " ", q)
    for m in re.finditer(r"fy\s*'?(\d{2,4})\b", q):
        y = m.group(1); ys.add(y if len(y) == 4 else "20" + y)
    q = re.sub(r"fy\s*'?(\d{2,4})\b", " ", q)
    ys.update(re.findall(r"\b(20\d{2})\b", q))
    return sorted(y for y in ys if y in YEARS)

def parse_query(q):
    ql = q.lower()
    company = next((c for c, names in ALIASES.items() if any(n in ql for n in names)), None)
    return company, find_years(q)

TOKEN = re.compile(r"[a-z0-9]+(?:[.,][0-9]+)*")
def tok(s): return TOKEN.findall(s.lower())
bm25 = BM25Okapi([tok(doc_text[i]) for i in ids])

comp_arr = np.array([docstore[i]["company"] for i in ids])
year_arr = np.array([docstore[i]["year"] for i in ids])

def filter_mask(company, years):
    m = np.ones(len(ids), bool)
    if company: m &= comp_arr == company
    if years:   m &= np.isin(year_arr, years)
    return m

def bm25_search(q, n=30, company=None, years=None):
    s = np.where(filter_mask(company, years), bm25.get_scores(tok(q)), -np.inf)
    return [ids[j] for j in np.argsort(-s)[:n] if np.isfinite(s[j])]

def rrf(*ranked_lists, k=60, top=30):
    s = {}
    for L in ranked_lists:
        for r, i in enumerate(L, 1):
            s[i] = s.get(i, 0) + 1 / (k + r)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])][:top]

def page_key(i):
    d = docstore[i]
    return f"{d['doc_id']}|{d['page']}"

## Building training data (without leaking the test set)

In [ ]:
# Holding out every page that answers a golden question, plus the whole FY2024 report
held_pages = {f"{r.gold_doc_id}|{p}" for r in answerable.itertuples() for p in r.gold_pages}
held_pages |= {page_key(k.split("|", 1)[1]) for k, v in rel.items()
               if v and k.split("|", 1)[1] in docstore}
TRAIN_YEARS = {"2023", "2025"}

def usable(i):
    d = docstore[i]
    if d["year"] not in TRAIN_YEARS or page_key(i) in held_pages:
        return False
    if d["type"] == "text":  return len(d["content"]) > 300
    return bool(d.get("summary")) and d["summary"] != "SKIP"

pool = [i for i in ids if usable(i)]
random.seed(7)
train_items = random.sample(pool, min(900, len(pool)))
print(f"held-out pages: {len(held_pages)} | usable items: {len(pool)} | sampled: {len(train_items)}")
print(pd.Series([docstore[i]["type"] for i in train_items]).value_counts())

In [ ]:
# Connecting to Gemini (Flash-Lite, pinned)
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
GEN_MODEL = "gemini-3.5-flash-lite"

def gemini_json(contents, model=GEN_MODEL, retries=5):
    for a in range(retries):
        try:
            r = client.models.generate_content(model=model, contents=contents,
                config=types.GenerateContentConfig(temperature=0.3, response_mime_type="application/json"))
            return json.loads(r.text)
        except json.JSONDecodeError:
            time.sleep(3)
        except Exception as ex:
            m = str(ex)
            if "503" in m or "UNAVAILABLE" in m: time.sleep(5 * 2 ** a)
            elif "429" in m or "RESOURCE_EXHAUSTED" in m:
                if "PerDay" in m: raise RuntimeError(f"Daily quota reached for {model}, rerun tomorrow")
                time.sleep(30)
            else: raise
    return None

In [ ]:
# Generating 2 questions per item with Gemini, 8 items per call (cached, resumable)
TQ_CACHE = EVAL_DIR / "train_questions.json"
tq = json.loads(TQ_CACHE.read_text()) if TQ_CACHE.exists() else {}

TQ_PROMPT = """You write search queries for training a retrieval model over Indian bank annual reports.
For EACH item below, write TWO different questions that this item alone answers:
1. a natural analyst question, e.g. "How much did gross NPAs rise in FY 2024-25?"
2. a short keyword-style query, e.g. "FY25 gross NPA closing balance"
Both must name the fiscal year from the item header. Never say "this table", "this chart" or a page number.
Paraphrase; do not copy long phrases. If an item has no real information, return an empty list for it.
Return ONLY JSON: [{"id": "I1", "questions": ["...", "..."]}, ...]"""

def header(d):
    y = int(d["year"])
    return f"{d['company'].replace('_', ' ')} annual report, FY {y - 1}-{str(y)[2:]} | {d['type']}"

def body(i):
    d = docstore[i]
    if d["type"] == "text":  return d["content"][:2500]
    if d["type"] == "table": return f"{d['summary']}\n{d['content'][:2000]}"
    return f"Chart/infographic: {d['summary']}"

todo = [i for i in train_items if i not in tq]
print(f"{len(todo)} items need questions (~{len(todo) // 8 + 1} calls)")
for b in tqdm(range(0, len(todo), 8)):
    short = {f"I{j + 1}": i for j, i in enumerate(todo[b:b + 8])}      # short IDs always match
    out = gemini_json([TQ_PROMPT] + [f"ID: {s}\n{header(docstore[i])}\n{body(i)}" for s, i in short.items()])
    if isinstance(out, dict): out = out.get("items", [out])
    for r in out or []:
        if isinstance(r, dict) and r.get("id") in short:
            tq[short[r["id"]]] = [q for q in r.get("questions", []) if isinstance(q, str) and len(q) > 8]
    TQ_CACHE.write_text(json.dumps(tq, ensure_ascii=False))
    time.sleep(4)
print(f"{len(tq)} items with questions, {sum(len(v) for v in tq.values())} questions total")

In [ ]:
# Building (question, passage) pairs and splitting train / validation by item
from datasets import Dataset

items_with_q = [i for i in train_items if tq.get(i)]
random.seed(11)
random.shuffle(items_with_q)
cut = int(len(items_with_q) * 0.9)

def pairs(items):
    return [{"anchor": QUERY_PREFIX + q, "positive": doc_text[i]} for i in items for q in tq[i]]

train_ds = Dataset.from_list(pairs(items_with_q[:cut]))
val_ds   = Dataset.from_list(pairs(items_with_q[cut:]))
print(f"train pairs: {len(train_ds)} | validation pairs: {len(val_ds)}")
for ex in train_ds.select(range(3)):
    print("\nQ:", ex["anchor"].replace(QUERY_PREFIX, ""), "\nP:", ex["positive"][:200])

## Baseline: the original model

In [ ]:
# Embedding every item with the base model, and searching with numpy (exact, no approximation)
from sentence_transformers import SentenceTransformer

def embed_all(model):
    return model.encode([doc_text[i] for i in ids], batch_size=64, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)

def dense_search(model, M, q, n=30, company=None, years=None):
    qv = model.encode([QUERY_PREFIX + q], normalize_embeddings=True, convert_to_numpy=True)[0]
    s = np.where(filter_mask(company, years), M @ qv, -np.inf)
    top = np.argpartition(-s, n)[:n]
    return [ids[j] for j in top[np.argsort(-s[top])] if np.isfinite(s[j])]

base = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
M_base = embed_all(base)
print(M_base.shape)

## Fine-tuning

In [ ]:
# Training with in-batch negatives: every other passage in the batch is a wrong answer
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

ft = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
args = SentenceTransformerTrainingArguments(
    output_dir="/content/ft_out",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,     # no repeated passage inside one batch
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)
trainer = SentenceTransformerTrainer(model=ft, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                                     loss=losses.MultipleNegativesRankingLoss(ft))
t0 = time.time()
trainer.train()
print(f"Training took {(time.time() - t0) / 60:.1f} min")

MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
ft.save(str(MODEL_DIR))
print("Saved to", MODEL_DIR)

In [ ]:
# Embedding every item with the fine-tuned model
M_ft = embed_all(ft)
np.save(PROJECT_DIR / "cache" / "vecs_finetuned.npy", M_ft)
print(M_ft.shape)

## Evaluating base vs fine-tuned

In [ ]:
# Defining the four setups: dense and hybrid, each with the base and fine-tuned model
def make_dense(model, M):
    def f(q):
        c, y = parse_query(q); return dense_search(model, M, q, 10, c, y)
    return f

def make_hybrid(model, M):
    def f(q):
        c, y = parse_query(q)
        return rrf(dense_search(model, M, q, 30, c, y), bm25_search(q, 30, c, y))[:10]
    return f

SETUPS = {
    "dense (base)":       make_dense(base, M_base),
    "dense (fine-tuned)": make_dense(ft, M_ft),
    "hybrid (base)":      make_hybrid(base, M_base),
    "hybrid (fine-tuned)": make_hybrid(ft, M_ft),
}
runs = {name: {r.qid: f(r.question) for r in tqdm(list(answerable.itertuples()), desc=name)}
        for name, f in SETUPS.items()}

In [ ]:
# Judging any new top-5 items the fine-tuned model surfaces (cached with Notebook 3's judgments)
REL_PROMPT = """You check search results for a question about a bank's annual report.
QUESTION: {q}
REFERENCE ANSWER: {a}
For EACH passage below, decide: does it contain the information needed to answer the question
correctly (consistent with the reference answer, for the right fiscal year)?
Merely mentioning the topic is NOT enough.
Return ONLY a JSON list: [{{"pid": "P1", "relevant": true}}, ...]"""

def passage(i):
    d = docstore[i]
    tag = f"[{d['type']}, FY{d['year']} report, p.{d['page']}]"
    return f"{tag} {d.get('summary', '')}" if d["type"] == "image" else f"{tag} {d['content'][:2500]}"

def is_gold(i, row):
    d = docstore[i]
    return d["doc_id"] == row.gold_doc_id and d["page"] in row.gold_pages

for row in tqdm(list(answerable.itertuples())):
    cand = {i for name in SETUPS for i in runs[name][row.qid][:5]}
    todo = sorted(i for i in cand if f"{row.qid}|{i}" not in rel and not is_gold(i, row))
    if not todo:
        continue
    pid = {f"P{k + 1}": i for k, i in enumerate(todo)}
    out = gemini_json([REL_PROMPT.format(q=row.question, a=row.answer)]
                      + [f"{p}: {passage(i)}" for p, i in pid.items()])
    if isinstance(out, dict): out = out.get("items", [out])
    for r in out or []:
        if isinstance(r, dict) and r.get("pid") in pid:
            rel[f"{row.qid}|{pid[r['pid']]}"] = bool(r.get("relevant"))
    (EVAL_DIR / "relevance_cache.json").write_text(json.dumps(rel))
    time.sleep(4)
print(f"{len(rel)} judgments cached")

In [ ]:
# Scoring: strict (gold page) and judged, overall and on the unseen FY2024 report
def first_hit(L, ok):
    return next((r for r, i in enumerate(L[:10], 1) if ok(i)), None)

recs = []
for name in SETUPS:
    for row in answerable.itertuples():
        L = runs[name][row.qid]
        s = first_hit(L, lambda i: is_gold(i, row))
        j = first_hit(L[:5], lambda i: is_gold(i, row) or rel.get(f"{row.qid}|{i}", False))
        recs.append({"setup": name, "qid": row.qid, "qtype": row.qtype,
                     "unseen_report": row.gold_doc_id.endswith("_2024"),
                     "hit@5": int(s is not None and s <= 5), "mrr": 1 / s if s else 0.0,
                     "j_hit@1": int(j == 1), "j_hit@5": int(j is not None), "j_mrr": 1 / j if j else 0.0})
res = pd.DataFrame(recs)
order = list(SETUPS)
cols = ["hit@5", "mrr", "j_hit@1", "j_hit@5", "j_mrr"]

print("   All 51 answerable questions    ")
display(res.groupby("setup")[cols].mean().round(3).loc[order])
n_unseen = res[res.unseen_report].qid.nunique()
print(f"    Unseen FY2024 report only ({n_unseen} questions)    ")
display(res[res.unseen_report].groupby("setup")[cols].mean().round(3).loc[order])
print("    Judged Hit@5 by question type    ")
display(res.pivot_table(index="setup", columns="qtype", values="j_hit@5").round(2).loc[order])
res.to_csv(EVAL_DIR / "finetune_results.csv", index=False)

In [ ]:
# Looking at questions the fine-tuned model won or lost (dense setups)
a = res[res.setup == "dense (base)"].set_index("qid")["j_hit@5"]
b = res[res.setup == "dense (fine-tuned)"].set_index("qid")["j_hit@5"]
won, lost = a[(a == 0) & (b == 1)].index, a[(a == 1) & (b == 0)].index
print(f"fine-tuning fixed {len(won)} questions and broke {len(lost)}")
q_by_id = answerable.set_index("qid").question
for label, group in [("FIXED", won), ("BROKE", lost)]:
    for qid in list(group)[:3]:
        print(f"[{label}] {q_by_id[qid]}")

In [ ]:
# Saving a Chroma collection with the fine-tuned embeddings (for the app in Notebook 6)
ft_client = chromadb.PersistentClient(path=CHROMA_LOCAL)
try:
    ft_client.delete_collection("annual_reports_ft")
except Exception:
    pass
ft_col = ft_client.create_collection("annual_reports_ft", metadata={"hnsw:space": "cosine"})
meta_all = col.get(include=["metadatas"])["metadatas"]
for b in range(0, len(ids), 1000):
    ft_col.add(ids=ids[b:b + 1000], embeddings=M_ft[b:b + 1000].tolist(),
               documents=[doc_text[i] for i in ids[b:b + 1000]], metadatas=meta_all[b:b + 1000])
shutil.rmtree(PROJECT_DIR / "chroma", ignore_errors=True)
shutil.copytree(CHROMA_LOCAL, PROJECT_DIR / "chroma")
print("Collections on Drive:", [getattr(c, "name", c) for c in ft_client.list_collections()])